In [26]:
!pip install numpy==1.21.0

In [27]:
# from google.colab import drive
# drive.mount('/content/drive')

# Pruning -> Fine-tune

1. Sort the weight of Batchnorm1d.
2. Remove the channel of Conv1d if the weight of Batchnorm1d before this channel is smaller than threshold.
3. Fine-tine model to recover accuracy.


## Load model

In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from speech_command_dataset import SpeechCommandDataset
from torchvision.transforms import Compose
import torchvision.models as models
import model

# from apex import amp

import os,time
import numpy as np
import matplotlib.pyplot as plt

In [29]:
torch.manual_seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [30]:
BATCH_SIZE = 4
training_params = {"batch_size": BATCH_SIZE,
                       "shuffle": True,
                       "drop_last": False,
                       "num_workers": 1}

testing_params = {"batch_size": BATCH_SIZE,
                       "shuffle": False,
                       "drop_last": False,
                       "num_workers": 1}

train_set = SpeechCommandDataset()
train_loader = DataLoader(train_set, **training_params)

test_set = SpeechCommandDataset(is_training=False)
test_loader = DataLoader(test_set, **testing_params)

In [31]:
# load model
net = model.SincNet().cuda()
model_path = 'C:/Users/lab423/Desktop/Deep Learn/DL_lab4/Checkpoint/SincNet_best.pth.tar'
# you could load the model after pruning and fine-tune to prune again
# model_path = './Checkpoint/SincNet_finetune.pth.tar'

if os.path.isfile(model_path):
    print("=> loading checkpoint '{}'".format(model_path))
    checkpoint = torch.load(model_path)
#     print(checkpoint)
    net.load_state_dict(checkpoint['state_dict'])
else:
    print("=> no checkpoint found at '{}'".format(model_path))

=> loading checkpoint 'C:/Users/lab423/Desktop/Deep Learn/DL_lab4/Checkpoint/SincNet_best.pth.tar'


C:\Users\lab423\AppData\Local\Temp\ipykernel_25784\347974298.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path)


In [32]:
print('Before pruning')

loss_func = nn.CrossEntropyLoss()

total_val_loss = 0
correct = 0
total = 0
batch_num = 0

net.eval()

for audios, labels in test_loader:
    audios = audios.cuda()
    labels = labels.cuda()

    outputs = net(audios)
    loss = loss_func(outputs, labels)
    total_val_loss += loss.item()
    batch_num += 1
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum()


val_loss = total_val_loss / batch_num
val_acc = 100.0 * float(correct) / float(total)


print('Validation loss: %.4f' % val_loss,'Validation accuracy: %.2f' % val_acc)

Before pruning
Validation loss: 0.5553 Validation accuracy: 83.75


## Start Pruning

In [33]:
# you can choose your pruning rate
pruning_rate = 0.1

In [34]:
# extract the weight of BatchNorm1d layer and sort them

total_par = sum([param.nelement() for param in net.parameters()])
print("Number of parameter: %.2fM" % (total_par/1e6))

total = 0

for m in net.modules():
    if isinstance(m, nn.BatchNorm1d):
        total += m.weight.data.shape[0]

bn = torch.zeros(total)
index = 0
for m in net.modules():
    if isinstance(m, nn.BatchNorm1d):
        size = m.weight.data.shape[0]
#         print(size)
        bn[index:(index+size)] = m.weight.data.abs().clone()

        index += size

y, i = torch.sort(bn)
thre_index = int(total * pruning_rate)
thre = y[thre_index]


Number of parameter: 0.27M


In [35]:
# record the renaming weight
pruned = 0
cfg = []
cfg_mask = []
for k, m in enumerate(net.modules()):
    if isinstance(m, nn.BatchNorm1d):
        weight_copy = m.weight.data.abs().clone().cpu()

        # if weight larger than threshold
        mask = weight_copy.gt(thre).float().cuda()

        # pruning number
        pruned = pruned + mask.shape[0] - torch.sum(mask)
        m.weight.data.mul_(mask)
        m.bias.data.mul_(mask)

        cfg.append(int(torch.sum(mask)))
        cfg_mask.append(mask.clone())
        print('layer index: {:d} \t total channel: {:d} \t remaining channel: {:d}'.format(k, mask.shape[0], int(torch.sum(mask))))

    elif isinstance(m, nn.AvgPool1d):
        cfg.append('P')


# print('cfg',cfg)
pruned_ratio = pruned/total

print('Pre-processing Successful!')

layer index: 4 	 total channel: 40 	 remaining channel: 39
layer index: 11 	 total channel: 256 	 remaining channel: 241
layer index: 17 	 total channel: 256 	 remaining channel: 236
layer index: 23 	 total channel: 256 	 remaining channel: 232
layer index: 29 	 total channel: 256 	 remaining channel: 193
layer index: 35 	 total channel: 160 	 remaining channel: 160
Pre-processing Successful!


In [36]:
new_module = model.SincNet(cfg).cuda()

In [37]:
old_modules = list(net.modules())
new_modules = list(new_module.modules())
layer_id_in_cfg = 0
start_mask = torch.ones(1)
end_mask = cfg_mask[layer_id_in_cfg]
conv_count = 0

for layer_id in range(len(old_modules)):
    m0 = old_modules[layer_id]
    m1 = new_modules[layer_id]
    if isinstance(m0, nn.BatchNorm1d):


        idx1 = np.squeeze(np.argwhere(np.asarray(end_mask.cpu().numpy())))

        m1.weight.data = m0.weight.data[idx1.tolist()].clone()
        m1.bias.data = m0.bias.data[idx1.tolist()].clone()
        m1.running_mean = m0.running_mean[idx1.tolist()].clone()
        m1.running_var = m0.running_var[idx1.tolist()].clone()
        layer_id_in_cfg += 1
        start_mask = end_mask.clone()
        if layer_id_in_cfg < len(cfg_mask):  # do not change in Final FC
            end_mask = cfg_mask[layer_id_in_cfg]
    elif isinstance(m0, nn.Conv1d):
        if isinstance(old_modules[layer_id-4], nn.BatchNorm1d) or isinstance(old_modules[layer_id-3], nn.BatchNorm1d) or isinstance(old_modules[layer_id+2], nn.BatchNorm1d):
            # This convers the convolutions in the residual block.
            conv_count += 1

            idx0 = np.squeeze(np.argwhere(np.asarray(start_mask.cpu().numpy())))
            idx1 = np.squeeze(np.argwhere(np.asarray(end_mask.cpu().numpy())))


            if conv_count % 2 != 0:
                w1 = m0.weight.data[idx0.tolist(), :, :].clone()
                print('In shape: {:d}, Out shape {:d}.'.format(idx0.size, idx0.size))
            else:
                w1 = m0.weight.data[:, idx0.tolist(), :].clone()
                w1 = w1[idx1.tolist(), :, :].clone()
                print('In shape: {:d}, Out shape {:d}.'.format(idx0.size, idx1.size))


            m1.weight.data = w1.clone()

            continue

        # We need to consider the case where there are downsampling convolutions.
        # For these convolutions, we just copy the weights.
        m1.weight.data = m0.weight.data.clone()
    elif isinstance(m0, nn.Linear):
        idx0 = np.squeeze(np.argwhere(np.asarray(start_mask.cpu().numpy())))
        if idx0.size == 1:
            idx0 = np.resize(idx0, (1,))

        m1.weight.data = m0.weight.data[:, idx0].clone()
        m1.bias.data = m0.bias.data.clone()

# print(cfg)
torch.save({'cfg': cfg, 'state_dict': new_module.state_dict()}, os.path.join('./Checkpoint', 'SincNet_prune.pth.tar'))

In shape: 39, Out shape 39.
In shape: 39, Out shape 241.
In shape: 241, Out shape 241.
In shape: 241, Out shape 236.
In shape: 236, Out shape 236.
In shape: 236, Out shape 232.
In shape: 232, Out shape 232.
In shape: 232, Out shape 193.
In shape: 193, Out shape 193.
In shape: 193, Out shape 160.


## Fine-tune

In [38]:
checkpoint = torch.load('./Checkpoint/SincNet_prune.pth.tar')
net = model.SincNet(cfg=checkpoint['cfg'])
net.load_state_dict(checkpoint['state_dict'])

net.cuda()
print(net)

SincNet(
  (sincconv): _Layer(
    (conv0): SincConv1d()
    (logabs): LogAbs()
    (bn): BatchNorm1d(39, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (pool): AvgPool1d(kernel_size=(2,), stride=(2,), padding=(0,))
  )
  (features): ModuleList(
    (0): _Layer(
      (conv0): Conv1d(39, 39, kernel_size=(25,), stride=(2,), groups=39)
      (conv1): Conv1d(39, 241, kernel_size=(1,), stride=(1,))
      (relu): ReLU()
      (bn): BatchNorm1d(241, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (pool): AvgPool1d(kernel_size=(2,), stride=(2,), padding=(0,))
    )
    (1): _Layer(
      (conv0): Conv1d(241, 241, kernel_size=(9,), stride=(1,), groups=241)
      (conv1): Conv1d(241, 236, kernel_size=(1,), stride=(1,))
      (relu): ReLU()
      (bn): BatchNorm1d(236, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (pool): AvgPool1d(kernel_size=(2,), stride=(2,), padding=(0,))
    )
    (2): _Layer(
      (conv0): Conv1d(236, 236,

C:\Users\lab423\AppData\Local\Temp\ipykernel_25784\1899999241.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('./Checkpoint/SincNet_prune.pth.tar

In [39]:
print('Before fine-tune')

total_val_loss = 0
correct = 0
total = 0
batch_num = 0

net.eval()

for audios, labels in test_loader:
    audios = audios.cuda()
    labels = labels.cuda()

    outputs = net(audios)
    loss = loss_func(outputs, labels)
    total_val_loss += loss.item()
    batch_num += 1
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum()


val_loss = total_val_loss / batch_num
val_acc = 100.0 * float(correct) / float(total)

print('Validation loss: %.4f' % val_loss,'Validation accuracy: %.2f' % val_acc)

Before fine-tune
Validation loss: 29.6177 Validation accuracy: 12.50


In [40]:
EPOCH = 10
LR = 1e-3
Weight_decay = 1e-9

In [41]:
optimizer = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=Weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5,  patience=15, verbose=True, eps=1e-09)

In [42]:
print('Begin fine-tune...')
best_accuracy = 0

for epoch in range(EPOCH):
    net.train()
    start_time = time.time()
    total_train_loss = 0
    correct = 0
    total = 0
    batch_num = 0

    for step, (audios, labels) in enumerate(train_loader):
        audios = audios.cuda()
        labels = labels.cuda()
        outputs = net(audios)

        loss = loss_func(outputs, labels)

        total_train_loss += loss.item()
        batch_num += 1
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum()


        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    end_time = time.time()
    train_time = end_time - start_time
    train_loss = total_train_loss / batch_num
    train_acc = 100.0 * float(correct) / float(total)



    print(time.strftime("%d %b %Y %H:%M:%S", time.localtime()))
    print('Epoch: %3d' % epoch, '|train loss: %.4f' % train_loss, '|train accuracy: %.2f' % train_acc,
          '|train time: %.2f' % train_time)

    scheduler.step(train_loss)

torch.save({'cfg': None, 'state_dict': net.state_dict()}, os.path.join('./Checkpoint', 'SincNet_finetune.pth.tar'))
print('Saving..')


Begin fine-tune...
09 Nov 2024 13:40:09
Epoch:   0 |train loss: 0.9248 |train accuracy: 71.08 |train time: 3.94
09 Nov 2024 13:40:13
Epoch:   1 |train loss: 0.6897 |train accuracy: 77.70 |train time: 3.82
09 Nov 2024 13:40:16
Epoch:   2 |train loss: 0.5971 |train accuracy: 81.69 |train time: 3.83
09 Nov 2024 13:40:20
Epoch:   3 |train loss: 0.5548 |train accuracy: 81.89 |train time: 3.83
09 Nov 2024 13:40:24
Epoch:   4 |train loss: 0.5594 |train accuracy: 82.77 |train time: 3.83
09 Nov 2024 13:40:28
Epoch:   5 |train loss: 0.5177 |train accuracy: 84.42 |train time: 3.83
09 Nov 2024 13:40:32
Epoch:   6 |train loss: 0.4956 |train accuracy: 85.39 |train time: 3.84
09 Nov 2024 13:40:36
Epoch:   7 |train loss: 0.4967 |train accuracy: 83.93 |train time: 3.87
09 Nov 2024 13:40:40
Epoch:   8 |train loss: 0.4614 |train accuracy: 86.17 |train time: 3.87
09 Nov 2024 13:40:43
Epoch:   9 |train loss: 0.4268 |train accuracy: 86.85 |train time: 3.86
Saving..


In [43]:
total_par = sum([param.nelement() for param in net.parameters()])
print("Number of parameter: %.2fM" % (total_par/1e6))

Number of parameter: 0.21M


In [44]:
print('After fine-tune')

total_val_loss = 0
correct = 0
total = 0
batch_num = 0

net.eval()

for audios, labels in test_loader:
    audios = audios.cuda()
    labels = labels.cuda()

    outputs = net(audios)
    loss = loss_func(outputs, labels)
    total_val_loss += loss.item()
    batch_num += 1
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum()


val_loss = total_val_loss / batch_num
val_acc = 100.0 * float(correct) / float(total)

print('Validation loss: %.4f' % val_loss,'Validation accuracy: %.2f' % val_acc)

After fine-tune
Validation loss: 0.7977 Validation accuracy: 80.00


### repeatedly for twice

In [45]:
# you can choose your pruning rate
pruning_rate = 0.1

In [46]:
# extract the weight of BatchNorm1d layer and sort them

total_par = sum([param.nelement() for param in net.parameters()])
print("Number of parameter: %.2fM" % (total_par/1e6))

total = 0

for m in net.modules():
    if isinstance(m, nn.BatchNorm1d):
        total += m.weight.data.shape[0]

bn = torch.zeros(total)
index = 0
for m in net.modules():
    if isinstance(m, nn.BatchNorm1d):
        size = m.weight.data.shape[0]
#         print(size)
        bn[index:(index+size)] = m.weight.data.abs().clone()

        index += size

y, i = torch.sort(bn)
thre_index = int(total * pruning_rate)
thre = y[thre_index]


Number of parameter: 0.21M


In [47]:
# record the renaming weight
pruned = 0
cfg = []
cfg_mask = []
for k, m in enumerate(net.modules()):
    if isinstance(m, nn.BatchNorm1d):
        weight_copy = m.weight.data.abs().clone().cpu()

        # if weight larger than threshold
        mask = weight_copy.gt(thre).float().cuda()

        # pruning number
        pruned = pruned + mask.shape[0] - torch.sum(mask)
        m.weight.data.mul_(mask)
        m.bias.data.mul_(mask)

        cfg.append(int(torch.sum(mask)))
        cfg_mask.append(mask.clone())
        print('layer index: {:d} \t total channel: {:d} \t remaining channel: {:d}'.format(k, mask.shape[0], int(torch.sum(mask))))

    elif isinstance(m, nn.AvgPool1d):
        cfg.append('P')


# print('cfg',cfg)
pruned_ratio = pruned/total

print('Pre-processing Successful!')


layer index: 4 	 total channel: 39 	 remaining channel: 36
layer index: 11 	 total channel: 241 	 remaining channel: 223
layer index: 17 	 total channel: 236 	 remaining channel: 203
layer index: 23 	 total channel: 232 	 remaining channel: 200
layer index: 29 	 total channel: 193 	 remaining channel: 168
layer index: 35 	 total channel: 160 	 remaining channel: 160
Pre-processing Successful!


In [48]:
new_module = model.SincNet(cfg).cuda()

In [49]:
old_modules = list(net.modules())
new_modules = list(new_module.modules())
layer_id_in_cfg = 0
start_mask = torch.ones(1)
end_mask = cfg_mask[layer_id_in_cfg]
conv_count = 0

for layer_id in range(len(old_modules)):
    m0 = old_modules[layer_id]
    m1 = new_modules[layer_id]
    if isinstance(m0, nn.BatchNorm1d):


        idx1 = np.squeeze(np.argwhere(np.asarray(end_mask.cpu().numpy())))

        m1.weight.data = m0.weight.data[idx1.tolist()].clone()
        m1.bias.data = m0.bias.data[idx1.tolist()].clone()
        m1.running_mean = m0.running_mean[idx1.tolist()].clone()
        m1.running_var = m0.running_var[idx1.tolist()].clone()
        layer_id_in_cfg += 1
        start_mask = end_mask.clone()
        if layer_id_in_cfg < len(cfg_mask):  # do not change in Final FC
            end_mask = cfg_mask[layer_id_in_cfg]
    elif isinstance(m0, nn.Conv1d):
        if isinstance(old_modules[layer_id-4], nn.BatchNorm1d) or isinstance(old_modules[layer_id-3], nn.BatchNorm1d) or isinstance(old_modules[layer_id+2], nn.BatchNorm1d):
            # This convers the convolutions in the residual block.
            conv_count += 1

            idx0 = np.squeeze(np.argwhere(np.asarray(start_mask.cpu().numpy())))
            idx1 = np.squeeze(np.argwhere(np.asarray(end_mask.cpu().numpy())))


            if conv_count % 2 != 0:
                w1 = m0.weight.data[idx0.tolist(), :, :].clone()
                print('In shape: {:d}, Out shape {:d}.'.format(idx0.size, idx0.size))
            else:
                w1 = m0.weight.data[:, idx0.tolist(), :].clone()
                w1 = w1[idx1.tolist(), :, :].clone()
                print('In shape: {:d}, Out shape {:d}.'.format(idx0.size, idx1.size))


            m1.weight.data = w1.clone()

            continue

        # We need to consider the case where there are downsampling convolutions.
        # For these convolutions, we just copy the weights.
        m1.weight.data = m0.weight.data.clone()
    elif isinstance(m0, nn.Linear):
        idx0 = np.squeeze(np.argwhere(np.asarray(start_mask.cpu().numpy())))
        if idx0.size == 1:
            idx0 = np.resize(idx0, (1,))

        m1.weight.data = m0.weight.data[:, idx0].clone()
        m1.bias.data = m0.bias.data.clone()

# print(cfg)
torch.save({'cfg': cfg, 'state_dict': new_module.state_dict()}, os.path.join('./Checkpoint', 'SincNet_prune.pth.tar'))

In shape: 36, Out shape 36.
In shape: 36, Out shape 223.
In shape: 223, Out shape 223.
In shape: 223, Out shape 203.
In shape: 203, Out shape 203.
In shape: 203, Out shape 200.
In shape: 200, Out shape 200.
In shape: 200, Out shape 168.
In shape: 168, Out shape 168.
In shape: 168, Out shape 160.


In [50]:
checkpoint = torch.load('./Checkpoint/SincNet_prune.pth.tar')
net = model.SincNet(cfg=checkpoint['cfg'])
net.load_state_dict(checkpoint['state_dict'])

net.cuda()
print(net)

SincNet(
  (sincconv): _Layer(
    (conv0): SincConv1d()
    (logabs): LogAbs()
    (bn): BatchNorm1d(36, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (pool): AvgPool1d(kernel_size=(2,), stride=(2,), padding=(0,))
  )
  (features): ModuleList(
    (0): _Layer(
      (conv0): Conv1d(36, 36, kernel_size=(25,), stride=(2,), groups=36)
      (conv1): Conv1d(36, 223, kernel_size=(1,), stride=(1,))
      (relu): ReLU()
      (bn): BatchNorm1d(223, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (pool): AvgPool1d(kernel_size=(2,), stride=(2,), padding=(0,))
    )
    (1): _Layer(
      (conv0): Conv1d(223, 223, kernel_size=(9,), stride=(1,), groups=223)
      (conv1): Conv1d(223, 203, kernel_size=(1,), stride=(1,))
      (relu): ReLU()
      (bn): BatchNorm1d(203, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (pool): AvgPool1d(kernel_size=(2,), stride=(2,), padding=(0,))
    )
    (2): _Layer(
      (conv0): Conv1d(203, 203,

C:\Users\lab423\AppData\Local\Temp\ipykernel_25784\1899999241.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('./Checkpoint/SincNet_prune.pth.tar

In [51]:
print('Before fine-tune')

total_val_loss = 0
correct = 0
total = 0
batch_num = 0

net.eval()

for audios, labels in test_loader:
    audios = audios.cuda()
    labels = labels.cuda()

    outputs = net(audios)
    loss = loss_func(outputs, labels)
    total_val_loss += loss.item()
    batch_num += 1
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum()


val_loss = total_val_loss / batch_num
val_acc = 100.0 * float(correct) / float(total)

print('Validation loss: %.4f' % val_loss,'Validation accuracy: %.2f' % val_acc)

Before fine-tune
Validation loss: 18.5469 Validation accuracy: 25.62


In [52]:
EPOCH = 10
LR = 1e-3
Weight_decay = 1e-9

In [ ]:
optimizer = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=Weight_decay)
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5,  patience=15, verbose=True, eps=1e-09)

In [54]:
print('Begin fine-tune...')
best_accuracy = 0

for epoch in range(EPOCH):
    net.train()
    start_time = time.time()
    total_train_loss = 0
    correct = 0
    total = 0
    batch_num = 0

    for step, (audios, labels) in enumerate(train_loader):
        audios = audios.cuda()
        labels = labels.cuda()
        outputs = net(audios)

        loss = loss_func(outputs, labels)

        total_train_loss += loss.item()
        batch_num += 1
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum()


        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    end_time = time.time()
    train_time = end_time - start_time
    train_loss = total_train_loss / batch_num
    train_acc = 100.0 * float(correct) / float(total)



    print(time.strftime("%d %b %Y %H:%M:%S", time.localtime()))
    print('Epoch: %3d' % epoch, '|train loss: %.4f' % train_loss, '|train accuracy: %.2f' % train_acc,
          '|train time: %.2f' % train_time)

    scheduler.step(train_loss)

torch.save({'cfg': None, 'state_dict': net.state_dict()}, os.path.join('./Checkpoint', 'SincNet_finetune.pth.tar'))
print('Saving..')


Begin fine-tune...
09 Nov 2024 13:40:51
Epoch:   0 |train loss: 0.5229 |train accuracy: 84.42 |train time: 3.75
09 Nov 2024 13:40:55
Epoch:   1 |train loss: 0.4143 |train accuracy: 87.44 |train time: 3.70
09 Nov 2024 13:40:58
Epoch:   2 |train loss: 0.3892 |train accuracy: 89.19 |train time: 3.73
09 Nov 2024 13:41:02
Epoch:   3 |train loss: 0.4044 |train accuracy: 88.41 |train time: 3.72
09 Nov 2024 13:41:06
Epoch:   4 |train loss: 0.3721 |train accuracy: 88.80 |train time: 3.73
09 Nov 2024 13:41:09
Epoch:   5 |train loss: 0.3589 |train accuracy: 88.32 |train time: 3.74
09 Nov 2024 13:41:13
Epoch:   6 |train loss: 0.3708 |train accuracy: 87.93 |train time: 3.71
09 Nov 2024 13:41:17
Epoch:   7 |train loss: 0.3192 |train accuracy: 90.26 |train time: 3.71
09 Nov 2024 13:41:21
Epoch:   8 |train loss: 0.2775 |train accuracy: 92.21 |train time: 3.71
09 Nov 2024 13:41:24
Epoch:   9 |train loss: 0.3016 |train accuracy: 90.46 |train time: 3.71
Saving..


In [55]:
total_par = sum([param.nelement() for param in net.parameters()])
print("Number of parameter: %.2fM" % (total_par/1e6))

Number of parameter: 0.17M


In [56]:
print('After fine-tune')

total_val_loss = 0
correct = 0
total = 0
batch_num = 0

net.eval()

for audios, labels in test_loader:
    audios = audios.cuda()
    labels = labels.cuda()

    outputs = net(audios)
    loss = loss_func(outputs, labels)
    total_val_loss += loss.item()
    batch_num += 1
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum()


val_loss = total_val_loss / batch_num
val_acc = 100.0 * float(correct) / float(total)

print('Validation loss: %.4f' % val_loss,'Validation accuracy: %.2f' % val_acc)

After fine-tune
Validation loss: 0.5172 Validation accuracy: 87.50
